# Exploring Novae results

In [1]:
# %% Libraries
import os
import scanpy as sc
#import scvi
import numpy as np
from pathlib import Path
import pandas as pd
import anndata as ad
import rapids_singlecell as rsc
import novae

In [1]:
import os, torch
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("device_count:", torch.cuda.device_count())
print("current_device:", torch.cuda.current_device())
print("name(current):", torch.cuda.get_device_name(torch.cuda.current_device()))


CUDA_VISIBLE_DEVICES: None
device_count: 4
current_device: 0
name(current): NVIDIA A100-SXM4-80GB


In [3]:
# %% Setting Paths
MAIN_DIR_NAME = "cosmx_gray"
MAIN_DIR = next(p for p in Path.cwd().parents if (p / MAIN_DIR_NAME).exists()) / MAIN_DIR_NAME
os.chdir(MAIN_DIR)

# %% Setting Seed
SEED_VALUE = 42
# set NumPy RNG for consistency
np.random.seed(SEED_VALUE)

# %% object versions
CUR_OBJ_V = 'v9'
NEW_OVJ_V = 'v10'
CUR_OBJ_PATH = MAIN_DIR / 'data' / 'old_comb' / 'h5ad' / f'comb_{CUR_OBJ_V}.h5ad'
NEW_OBJ_PATH = MAIN_DIR / 'data' / 'old_comb' / 'h5ad' / f'comb_{NEW_OVJ_V}.h5ad'

# %% Load the unintegrated object
comb = sc.read_h5ad(CUR_OBJ_PATH)

In [4]:
model1 = novae.Novae.from_pretrained("data/old_comb/novae_model")
model1

Loading weights from local directory


Novae model
   ├── Known genes: 60697
   ├── Parameters: 32.1M
   ├── Model name: data/old_comb/novae_model
   ├── Trained: True
   └── Multimodal: False

In [5]:
comb

AnnData object with n_obs × n_vars = 1326786 × 6175
    obs: 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD68', 'Max.CD68', 'Mean.Membrane', 'Max.Membrane', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'SplitRatioToLocal', 'NucArea', 'NucAspectRatio', 'Circularity', 'Eccentricity', 'Perimeter', 'Solidity', 'assay_type', 'version', 'Run_Tissue_name', 'Panel', 'cellSegmentationSetId', 'cellSegmentationSetName', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'unassignedTranscripts', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 'negprobes_quantile_0.85', 'negprobes_quantile_0.9', 'negprobes_quantile_0.95', 'negprobes_quantile_0.99', 'nCount_negprobes', 'nFeature_negprobes', 'median_falsecode', 'falsecode_quantile_0.75', 'falsecode_quantile_0.8', 'f

## UMAP from novae spatial domains

In [8]:
rsc.pp.neighbors(comb, use_rep="novae_latent_corrected", key_added="nb_novae", random_state=SEED_VALUE, n_neighbors=15)


CUDADriverError: CUDA_ERROR_OUT_OF_MEMORY: out of memory

In [13]:
sc.tl.umap(comb, neighbors_key="nb_novae", random_state=SEED_VALUE)

KeyboardInterrupt: 

In [6]:
# setting constants
NOVAE_NEIGHBORS_KEY = "spatial_connectivities"
NOVAE_UMAP_KEY = "umap_novae"

sc.tl.umap(
    comb, 
    neighbors_key=NOVAE_NEIGHBORS_KEY,
    key_added=NOVAE_UMAP_KEY,
    min_dist=0.2,
    spread=2.0,
    random_state=SEED_VALUE,
)

ValueError: Did not find .uns['spatial_connectivities']. Run `sc.pp.neighbors` first.

In [ ]:
# setting constants
NOVAE_NEIGHBORS_KEY = "spatial_neighbors"

# visualize clusters

sc.pl.umap(
    comb,
    color=["novae_domains_7"],
    size=0.5,
    frameon=False,
    legend_loc='on data',
    return_fig = False,
    show=False,
)